<div align="right" style="text-align: right"><i>Peter Norvig<br>April 2026</i></div>

# Did you solve it? R y clvr ngh t rd ths sntnc?

Alex Bellos's [30 March 2026 column](https://www.theguardian.com/science/2026/mar/30/did-you-solve-it-r-y-clvr-ngh-t-rd-ths-sntnc) asks us to guess famous phrases or sayings, given the shapes of the rectangles that bound the letters, and the clue that vowels are <span style="color:green">**green**</span> and consonants are <span style="color:blue">**blue**</span>.  

Here's one of the puzzles:

![](https://i.guim.co.uk/img/media/dd62fe8dfdc6eb9d98815a2e14791ae268aa4d46/0_0_580_75/master/580.jpg?width=310&dpr=2&s=none&crop=none)

I can help solve this problem by using code to constrain what each letter and each word might be.

I'll start by defining different subsets of letters: `v` for the vowels, `c` for the consonants, `a` for the letters whose shape ascends above the norm, etc.:

In [24]:
letters = set('abcdefghijklmnopqrstuvwxyz')
v = set('aeiou')    # vowels (green)
c = letters - v     # consonants (blue)

a = set('bdfhijlt') # ascending
d = set('gjpqy')    # descending
b = letters - a - d # block: neither ascending nor descending

t = set('li')       # thin
w = set('mw')       # wide
n = letters - t - w # normal: neither wide nor thin

Now I can say that the first letter of the first word above (the green rectangle) is a block-shaped vowel; the intersection of the **b**lock and **v**owel sets, denoted in Python with the `&` operator:

In [2]:
b&v

{'a', 'e', 'o', 'u'}

The first word has three letters, `b&v` followed by two **a**scending **t**hin **c**onsonants, `a&t&c`:

In [3]:
[b&v, a&t&c, a&t&c]

[{'a', 'e', 'o', 'u'}, {'l'}, {'l'}]

Neat! There is only one **a**scending **t**hin **c**onsonant, `'l'`.

The whole puzzle is as follows (I made the apostrophe be a word of its own):

In [4]:
puzzle3 = [[b&v, a&t&c, a&t&c], [a&c, a&c, b&v], [w&c, b&v, b&c, a&c, a&c], [{"’"}], [b&c], [b&v], [b&c, a&c, b&v, d&c, b&v]]
puzzle3

[[{'a', 'e', 'o', 'u'}, {'l'}, {'l'}],
 [{'b', 'd', 'f', 'h', 'j', 'l', 't'},
  {'b', 'd', 'f', 'h', 'j', 'l', 't'},
  {'a', 'e', 'o', 'u'}],
 [{'m', 'w'},
  {'a', 'e', 'o', 'u'},
  {'c', 'k', 'm', 'n', 'r', 's', 'v', 'w', 'x', 'z'},
  {'b', 'd', 'f', 'h', 'j', 'l', 't'},
  {'b', 'd', 'f', 'h', 'j', 'l', 't'}],
 [{'’'}],
 [{'c', 'k', 'm', 'n', 'r', 's', 'v', 'w', 'x', 'z'}],
 [{'a', 'e', 'o', 'u'}],
 [{'c', 'k', 'm', 'n', 'r', 's', 'v', 'w', 'x', 'z'},
  {'b', 'd', 'f', 'h', 'j', 'l', 't'},
  {'a', 'e', 'o', 'u'},
  {'g', 'j', 'p', 'q', 'y'},
  {'a', 'e', 'o', 'u'}]]

## Possible Words

What combinations of letters can each word pattern make? The function `possible_words` goes through the pattern one letter set at a time and builds up all possible ways of adding each possible letter to each possible partial word string:

In [5]:
def possible_words(pattern: list[set[str]]) -> set[str]:
    """All ways of choosing one letter from each of the possible letters in the word pattern."""
    words = {''} # To start there is one possible partial word, with no letters
    for letter_set in pattern:
        # On each turn, add each possible letter to each possible partial word
        words = {word + letter for word in words for letter in letter_set}
    return words

For example,

In [6]:
possible_words([b&v, a&t&c, a&t&c])

{'all', 'ell', 'oll', 'ull'}

Another example:

In [7]:
possible_words([{'b', 'c'}, {'a', 'o'}, {'n', 't'}])

{'ban', 'bat', 'bon', 'bot', 'can', 'cat', 'con', 'cot'}

Let's trace through how `possible_words` works on this example. It starts with one possible partial word, the empty string:

    words = {''}

Then it enterts the `for` loop and looks at the first letter set, `{'b', 'c'}`, and adds each letter to each partial word (just one of them: the empty string) to get a set of two partial words:

    words = {'b', 'c'}

It does the same thing with the second letter set, `{'a', 'o'}`, to get a set of four partial words:

    words = {'ba', 'bo', 'ca', 'co'}

Finally, it considers the third letter set, `{'n', 't'}`, and gets the final answer, a set of eight words:

    words = {'ban', 'bat', 'bon', 'bot', 'can', 'cat', 'con', 'cot'}

## Dictionary Words

What actual dictionary words could a pattern stand for? To answer that I'll need a list of actual dictionary words. Furthermore, when `possible_words(pattern)` returns more than one word, I'll have to pick one. To facilitate that, I will use a word list that includes the frequency of each word so that I can pick the word with the highest frequency. 

I'll download the word list file "[count_big.txt](count_big.txt)" which has the format: 

    a           21160
    aah         1
    aaron       5
    ab          2
    aback       3
    abacus      1
    abandon     32
    abandoned   72
    abandoning  27
    abandonment 15

A few things to note about the following command (but you don't need to memorize):
-  The `!` at the start of a line means to do an operating system command, not a Python command.
-  The `[ -e count_big.txt ] ||` part says to skip downloading the file if it already exists.
-  The `curl` command downloads the file


In [8]:
! [ -e count_big.txt ] || curl -O https://norvig.com/ngrams/count_big.txt

I'll read the contents of the file into a Python dictionary, which will have the form `{'a': 21160, 'aah': 1, ...}`. 

In [9]:
def make_dictionary(lines) -> dict:
    """The lines are strings with a word and a frequency count; make it into a dict."""
    counts = {} # Start with an emoty dict
    for line in lines:
        word, count = line.split()
        counts[word] = int(count)
    return counts

dictionary = make_dictionary(open('count_big.txt'))

Here are some things you can do with the dictionary:

In [10]:
'aback' in dictionary

True

In [11]:
'the' in dictionary

True

In [12]:
'xyzzy!@#$' in dictionary

False

In [13]:
dictionary['the'] # get the frequency count

80030

In [14]:
dictionary.get('the', 0) # Get the count, with a default of 0 if word is not in dictionary

80030

In [23]:
dictionary.get('xyzzy!@#$', 0) # Get the count, with a default of 0 if word is not in dictionary

0

In [15]:
len(dictionary) # the number of words in the dictionary

29136

Now I want to take a pattern  and figure out the most likely word (which I will guess is the most frequent one):

In [16]:
def most_likely_word(pattern: list[set[str]]) -> str:
    """Out of all the possible words the pattern can make, pick the most frequent one."""
    return max(possible_words(pattern), key=frequency)

def frequency(word) -> int: 
    """The frequency count of the word in the dictionary, or 0 by default."""
    return dictionary.get(word, 0) 

In [17]:
most_likely_word([b&v, a&t&c, a&t&c])

'all'

So far, so good!

A puzzle consists of a list of word patterns, and we can generate a best guess at solving the puzzle by finding the `most_likely_word` for each word pattern, and then joining them into a big string.

In [18]:
def solve(puzzle: list[list[set[str]]]) -> str:
    """Given a puzzle (a list of word patterns), return a string formed from the most likely matching words."""
    return ' '.join(most_likely_word(pattern) for pattern in puzzle)

## Puzzle #3

We're ready to see if our program can solve the puzzle:

![](https://i.guim.co.uk/img/media/dd62fe8dfdc6eb9d98815a2e14791ae268aa4d46/0_0_580_75/master/580.jpg?width=310&dpr=2&s=none&crop=none)



In [19]:
solve(puzzle3)                                 

'all the world ’ s a stage'

It worked! Let's do another one:

## Puzzle #1

![](https://i.guim.co.uk/img/media/a54abfc70c03cf9bc40e178c4e6186915b97ea1e/0_0_580_75/master/580.jpg?width=310&dpr=2&s=none&crop=none)

In [20]:
solve([[b&v, a&t&c, a&t&c], [{"’"}], [b&c], [w&c, b&v, t&c, t&c], [a&c, a&c, b&v, a&c], [b&v, b&c, a&c, b&c], [w&c, b&v, t&c, t&c]])

'all ’ s well that ends well'

That's correct!

## Puzzle #8

 ![](https://i.guim.co.uk/img/media/3eba3f724bda55abf76c8bca57f645cd5a669aef/0_0_580_75/master/580.jpg?width=310&dpr=2&s=none&crop=none)

In [21]:
solve([[v, a&c, a&c], [b&n&c, b&n&v, b&n&v, a&c, b&c], [a&t&c, b&v, b&v, a&c],[a&c, b&v], [c, b&v, w&b&c, b&v]])

'all roads lead to some'

OK, not quite right, but a good hint.

One more:

## Puzzle #10


![](https://i.guim.co.uk/img/media/c358352e9f14f1fb0d0f458f034bb86777efcc83/0_0_464_75/master/464.jpg?width=310&dpr=2&s=none&crop=none)

In [22]:
solve([[a&c, b&v, b&c, b&v], [a&v, b&c], [a&c, a&t&c, a&v, b&c, a&c]])

'have in blind'

That's not right either, but again it is a good clue to the right answer. (One reason I didn't get this one right is that I didn't consider capital letters, and a Capital "L" has a different shape than a lowercase "l".)